# Question 1: Connect Four with fixed-depth Minimax

**Course:** AIMLZG557 Artificial and Computational Intelligence  
**Assignment:** Situated Learning, Question 1, group submission (Group 109)

**Contributors**

| Name | BITS ID | Contribution |
|:--|:--|--:|
| Stuti Srivastava | 2025AG05747 | 100% |
| Chirag Bhatnagar | 2025AG05066 | 100% |
| Aditya Saraswat | 2025AG05058 | 100% |
| Vanshika Sangal | 2025AG05381 | 100% |
| Neelam Devi | 2025AG05908 | 100% |

The human plays X and the computer plays O on a board with 7 columns and 6 rows.
The human moves first. The computer uses Minimax at a fixed depth of 3 disc drops,
with alpha-beta pruning and a hand-designed evaluation function. No machine
learning model or external package is used.

### Where each part of the question is answered

| Part | What the question asks | Sections in this notebook |
|:--|:--|:--|
| (a) | Choose a static evaluation function, justify it and explain it with a sample game state | Part (a): Static evaluation design and choice of weights; Part (a): Worked sample game state |
| (b) | One human player gives live input, the computer plays fixed-depth Minimax, turns alternate and the board is shown after each move | Part (b): Human input and alternating game loop; Part (b): Complete sample game; Part (b): Play against the computer |
| (c) | Python code for the design in part (a), using Minimax | Part (c): Board representation and game design; Part (c): Fixed-depth Minimax search; Verification |

## Part (c): Board representation and game design

The board is a list of 6 row lists, each containing 7 integers: 0 for an empty
cell, 1 for the human and 2 for the computer. Row 0 is the top; row 5 is the bottom.
The interface uses columns 1 to 7 and the implementation uses indices 0 to 6.
Dropping a disc scans upward from the bottom and occupies the first empty cell.

The program precomputes all 69 four-cell windows: 24 horizontal, 21 vertical and
12 in each diagonal direction. These windows support both win detection and
static evaluation. A full top row means there are no legal moves because normal
play always preserves gravity.

After each valid move, the program displays the board, checks for a win, then
checks for a draw. Invalid input and full columns prompt the human again without
changing the board or consuming the turn.

### Constants and precomputed windows

In [1]:
"""Connect Four with a fixed three-ply minimax computer player."""

from dataclasses import dataclass
from math import inf

ROWS, COLS = 6, 7
EMPTY, HUMAN, COMPUTER = 0, 1, 2
SEARCH_DEPTH = 3
WIN_SCORE = 1_000_000
COLUMN_ORDER = (3, 2, 4, 1, 5, 0, 6)

# Each of the 69 possible four-cell lines is constructed once.
WINDOWS = tuple(
    tuple((r + k * dr, c + k * dc) for k in range(4))
    for r in range(ROWS)
    for c in range(COLS)
    for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1))
    if 0 <= r + 3 * dr < ROWS and 0 <= c + 3 * dc < COLS
)

### Board operations

In [2]:
def new_board():
    return [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]


def legal_columns(board):
    return [c for c in COLUMN_ORDER if board[0][c] == EMPTY]


def drop_disc(board, column, player):
    """Mutate the board and return the landing row. Columns here are 0 to 6."""
    if player not in (HUMAN, COMPUTER):
        raise ValueError("Unknown player.")
    if not 0 <= column < COLS or board[0][column] != EMPTY:
        raise ValueError("Column is outside the board or full.")
    for row in range(ROWS - 1, -1, -1):
        if board[row][column] == EMPTY:
            board[row][column] = player
            return row
    raise ValueError("Column is full.")


def has_won(board, player):
    return any(all(board[r][c] == player for r, c in w) for w in WINDOWS)


def display_board(board):
    symbols = {EMPTY: ".", HUMAN: "X", COMPUTER: "O"}
    print("  1 2 3 4 5 6 7")
    for row in board:
        print("| " + " ".join(symbols[v] for v in row) + " |")
    print("+---------------+")

## Part (a): Static evaluation design and choice of weights

A positive score favours the computer. A negative score favours the human.
Terminal positions are handled before static evaluation: a computer win scores
1,000,000 plus the remaining depth; a human win scores -1,000,000 minus the
remaining depth; a draw scores 0. The remaining-depth term prefers an earlier
win and delays an unavoidable loss within the search horizon.

For each four-cell window, the heuristic uses the following weights. All cells
other than the listed discs must be empty.

| Pattern | Computer discs | Human discs |
|:--|--:|--:|
| One disc and three empty cells | +2 | -2 |
| Two discs and two empty cells | +12 | -15 |
| Three discs and one empty cell | +120 | -150 |
| Both players occur in the window | 0 | 0 |

Each computer disc in the centre column adds 6; each human disc there subtracts 6.

### Why these weights

**Only open windows count.** A window can still become a four only if it holds
discs of one player. A mixed window is dead for both players, so it scores 0.

**Each level is worth far more than the level below.** A single disc scores 2,
a pair 12 (6 times a single) and a three 120 (10 times a pair). A three needs
one more disc to win and a pair needs two, so the value rises steeply with the
number of discs. One open three is worth as much as ten open pairs, so the
search will not give up a real threat for a few loose pairs. Singles are worth
little and are equal for both sides; they mainly reward discs that sit in many
open windows.

**Human pairs and threes weigh 25% more (15 and 150 against 12 and 120).** A
depth-3 search ends after a computer move, a human move and a computer move, so
at every non-terminal leaf it is the human's turn. A human three at a leaf can
be completed on the very next move, while a computer three must wait a turn.
The larger penalty reflects that urgency and makes the computer prefer defence
in otherwise similar positions.

**Centre bonus of 6 per disc.** A cell in column 4 lies in 7 to 13 of the 69
windows, while a cell in column 1 or 7 lies in only 3 to 5. Centre discs take
part in the most possible fours and spoil the most lines for the opponent. The
single-disc weight already favours such cells; the bonus adds a fixed
preference for the centre column itself. At half the value of a pair, it breaks
ties toward the centre without outweighing any pair or three.

**Wins always dominate.** The maximum absolute window contribution is 150, so a
conservative nonterminal bound is 69 * 150 + 6 * 6 = 10,386. This is far below
1,000,000, so no heuristic score can outrank a detected win or loss.

**Known limit.** The heuristic does not check whether the empty cell of a three
can be played now (supported from below) or only later. It is a positional
estimate; legal move simulation and terminal checks decide actual tactical
wins. A worked example on a real board follows the search code.

### Static evaluation

In [3]:
def window_score(values):
    ai, human = values.count(COMPUTER), values.count(HUMAN)
    if ai and human:
        return 0
    # Four-in-a-row is handled by the terminal test, before this heuristic.
    return {0: 0, 1: 2, 2: 12, 3: 120, 4: 0}[ai] - {
        0: 0, 1: 2, 2: 15, 3: 150, 4: 0
    }[human]


def evaluate(board):
    """Static score from the computer's perspective; no learned model."""
    center = [board[r][COLS // 2] for r in range(ROWS)]
    score = 6 * (center.count(COMPUTER) - center.count(HUMAN))
    return score + sum(window_score([board[r][c] for r, c in w]) for w in WINDOWS)

## Part (c): Fixed-depth Minimax search

Depth 3 means a computer move, a human reply and another computer move. The
computer maximizes the score and the human is assumed to minimize it. Terminal
states stop early; otherwise the static evaluator is called only at depth 0.
The depth is fixed by SEARCH_DEPTH and is not adjusted during gameplay.

The search tries centre columns first in the order 4, 3, 5, 2, 6, 1, 7. Alpha-beta
pruning skips branches that cannot change the decision. It preserves the same
Minimax value as an exhaustive search at depth 3. Equal scores keep the first
column in that order, making runs reproducible. Each tentative disc is removed
in a finally block, so searching leaves the live board unchanged.

**Search procedure**

1. Return the terminal score if either player has won or the board is full.
2. Return the static score if no depth remains.
3. Try each legal move, recurse with depth reduced by one, and undo the disc.
4. Keep the maximum or minimum score for the side to move.
5. Update alpha or beta and stop the branch when alpha is at least beta.

At branching factor b and depth d, worst-case search time is O(b^d) for this
fixed-size board. Here b is at most 7 and d is 3. An unpruned, nonterminal search
visits at most 1 + 7 + 49 + 343 = 400 nodes including the root. Pruning usually
reduces this count, though its benefit depends on move order. Search reuses the
board and needs O(d) recursion space; the board and 69 windows have fixed size.

### Minimax with alpha-beta pruning

In [4]:
@dataclass
class SearchStats:
    nodes: int = 0
    leaves: int = 0
    cutoffs: int = 0


def minimax(board, depth, alpha, beta, maximizing, stats):
    """Return (score, column), searching at most depth further disc drops."""
    stats.nodes += 1
    if has_won(board, COMPUTER):
        stats.leaves += 1
        return WIN_SCORE + depth, None
    if has_won(board, HUMAN):
        stats.leaves += 1
        return -WIN_SCORE - depth, None
    moves = legal_columns(board)
    if not moves:
        stats.leaves += 1
        return 0, None
    if depth == 0:
        stats.leaves += 1
        return evaluate(board), None

    best_score = -inf if maximizing else inf
    best_column = None
    player = COMPUTER if maximizing else HUMAN
    for column in moves:
        row = drop_disc(board, column, player)
        try:
            score, _ = minimax(board, depth - 1, alpha, beta, not maximizing, stats)
        finally:
            board[row][column] = EMPTY
        if best_column is None or (score > best_score if maximizing else score < best_score):
            best_score, best_column = score, column
        if maximizing:
            alpha = max(alpha, best_score)
        else:
            beta = min(beta, best_score)
        if alpha >= beta:
            stats.cutoffs += 1
            break
    return best_score, best_column


def choose_computer_move(board):
    """Always use depth 3, stopping earlier only at terminal states."""
    stats = SearchStats()
    score, column = minimax(board, SEARCH_DEPTH, -inf, inf, True, stats)
    return column, score, stats

## Part (b): Human input and alternating game loop

`read_human_column` keeps asking until the human enters a column from 1 to 7
that is not full. Invalid input and full columns do not use up the turn.
`play_game` alternates the human and the computer, prints the board after
every move, then checks for a win and then for a draw.

In [5]:
def read_human_column(board, input_fn=input):
    while True:
        text = input_fn("Choose column 1 to 7, or q to quit: ").strip().lower()
        if text in ("q", "quit"):
            return None
        try:
            column = int(text) - 1
        except ValueError:
            print("Enter a whole number from 1 to 7.")
            continue
        if not 0 <= column < COLS:
            print("Choose a column from 1 to 7.")
        elif board[0][column] != EMPTY:
            print("That column is full. Choose another column.")
        else:
            return column


def play_game(input_fn=input):
    """One human (X) plays first, against a minimax computer (O)."""
    board, player = new_board(), HUMAN
    print("Connect Four: you are X; the computer is O. You move first.")
    display_board(board)
    while True:
        if player == HUMAN:
            try:
                column = read_human_column(board, input_fn)
            except (EOFError, KeyboardInterrupt):
                print("\nGame stopped.")
                return "quit", board
            if column is None:
                print("Game stopped.")
                return "quit", board
            print(f"You chose column {column + 1}.")
        else:
            column, score, stats = choose_computer_move(board)
            print(f"Computer chose column {column + 1} "
                  f"(depth {SEARCH_DEPTH}, nodes {stats.nodes}, score {score}).")
        drop_disc(board, column, player)
        display_board(board)
        if has_won(board, player):
            result = "human" if player == HUMAN else "computer"
            print("You win!" if player == HUMAN else "Computer wins!")
            return result, board
        if not legal_columns(board):
            print("Draw: the board is full.")
            return "draw", board
        player = COMPUTER if player == HUMAN else HUMAN

## Part (a): Worked sample game state

The following state is reached by the legal sequence human 1, computer 7,
human 2, computer 7, human 3. It is now the computer's turn. The human threatens
to complete the bottom row in column 4. The output lists the static window
contributions and then the depth-3 score of each legal computer move. This
distinguishes the heuristic for the current state from the searched move value.

In [6]:
from collections import Counter

sample = new_board()
for i, col in enumerate([0, 6, 1, 6, 2]):
    drop_disc(sample, col, HUMAN if i % 2 == 0 else COMPUTER)
display_board(sample)
contributions = Counter(window_score([sample[r][c] for r, c in w]) for w in WINDOWS)
print("Nonzero window contributions:")
for weight, count in sorted(contributions.items()):
    if weight:
        print(f"  {count} windows at {weight:+d}: {count * weight:+d}")
center_score = 6 * sum((sample[r][3] == COMPUTER) - (sample[r][3] == HUMAN)
                       for r in range(ROWS))
print(f"Centre contribution: {center_score:+d}")
print(f"Current static score: {evaluate(sample):+d}")
print("Depth-3 scores for each possible computer move:")
for col in legal_columns(sample):
    row = drop_disc(sample, col, COMPUTER)
    try:
        value, _ = minimax(sample, 2, -inf, inf, False, SearchStats())
    finally:
        sample[row][col] = EMPTY
    print(f"  Column {col + 1}: {value:+d}")
col, score, stats = choose_computer_move(sample)
print(f"Chosen column: {col + 1}; score: {score:+d}; nodes: {stats.nodes}")
assert col == 3

  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . O |
| X X X . . . O |
+---------------+
Nonzero window contributions:
  1 windows at -150: -150
  1 windows at -15: -15
  7 windows at -2: -14
  5 windows at +2: +10
  1 windows at +12: +12
Centre contribution: +0
Current static score: -157
Depth-3 scores for each possible computer move:
  Column 4: +118
  Column 3: -1000001
  Column 5: -1000001
  Column 2: -1000001
  Column 6: -1000001
  Column 1: -1000001
  Column 7: -1000001
Chosen column: 4; score: +118; nodes: 62


### Reading the static score by hand

Cells are written cNhM: column N, the M-th cell from the bottom. Only the 15
windows below hold discs of a single player; the other 54 are empty and score 0.

| Windows | Contents | Pattern | Score |
|:--|:--|:--|--:|
| Bottom row c1h1 to c4h1 | X X X . | human three | -150 |
| Bottom row c2h1 to c5h1 | X X . . | human pair | -15 |
| Bottom row c3h1 to c6h1; columns 1, 2 and 3 (h1 to h4); diagonals c1h1 to c4h4, c2h1 to c5h4, c3h1 to c6h4 | one X each | 7 human singles | 7 × -2 = -14 |
| Column 7, c7h1 to c7h4 | O O . . | computer pair | +12 |
| Bottom row c4h1 to c7h1; second row c4h2 to c7h2; column 7 c7h2 to c7h5; diagonals c7h1 to c4h4, c7h2 to c4h5 | one O each | 5 computer singles | 5 × +2 = +10 |
| Centre column | empty | no centre discs | 0 |
| **Total** | | | **-157** |

The human's open three on the bottom row outweighs everything else, so the
static score is strongly negative. This is the urgency that the heavier human
weights encode. The static score only describes the current board; the search
decides the move. Column 4 is the only move that stops a human win on the next
turn. It scores +118, while every other column scores -1,000,001 (a human win
one ply later), so the computer blocks. The same search also takes an immediate
computer win when one is available, as verified below. Static scores for future
potential lines do not override a detected win or loss.

## Verification

These checks cover gravity, all four winning directions, invalid inputs, full
columns, draws, immediate wins and blocks, board restoration, and the actual
alternating game loop. Eight reproducible legal positions compare the pruned
value and chosen move with exhaustive Minimax at the same depth. A complete
sample game verifies that the loop stops and announces the winner.

In [7]:
from contextlib import redirect_stdout
from io import StringIO
from random import Random


def run_checks():
    checks = 0

    def check(condition, label):
        nonlocal checks
        assert condition, label
        checks += 1

    check(len(WINDOWS) == 69 and len(set(WINDOWS)) == 69, "69 unique windows")
    board = new_board()
    check(len(board) == 6 and all(len(row) == 7 for row in board), "6 by 7 array")
    check(drop_disc(board, 2, HUMAN) == 5, "gravity at bottom")
    check(drop_disc(board, 2, COMPUTER) == 4, "gravity stacks")
    check(board[5][2] == HUMAN and board[4][2] == COMPUTER, "stack contents")

    # Legal move sequences reach wins in each orientation. First winner is final.
    for label, moves in [
        ("horizontal", [0, 6, 1, 6, 2, 5, 3]),
        ("vertical", [0, 1, 0, 1, 0, 2, 0]),
        ("diagonal up right", [0, 1, 1, 2, 4, 2, 2, 3, 4, 3, 5, 3, 3]),
        ("diagonal up left", [6, 5, 5, 4, 2, 4, 4, 3, 2, 3, 1, 3, 3]),
    ]:
        board = new_board()
        for i, column in enumerate(moves):
            if i:
                assert not has_won(board, HUMAN) and not has_won(board, COMPUTER)
            drop_disc(board, column, HUMAN if i % 2 == 0 else COMPUTER)
        check(has_won(board, HUMAN), label)

    board = new_board()
    for player in (HUMAN, COMPUTER) * 3:
        drop_disc(board, 0, player)
    check(0 not in legal_columns(board), "full columns excluded")
    for invalid in (-1, 0, 7):
        try:
            drop_disc(board, invalid, HUMAN)
        except ValueError:
            check(True, "illegal drop rejected")
        else:
            check(False, "illegal drop accepted")
    attempts = iter(["abc", "0", "8", "1", "2"])
    with redirect_stdout(StringIO()):
        check(read_human_column(board, lambda prompt: next(attempts)) == 1,
              "invalid input recovers without consuming turn")
    check(read_human_column(board, lambda prompt: "q") is None, "quit input")

    # Both tactical positions are reached by alternating legal moves.
    for moves, target, label in [
        ([6, 0, 6, 1, 5, 2, 4], 3, "take immediate win"),
        ([0, 6, 1, 6, 2], 3, "block immediate human win"),
    ]:
        board = new_board()
        for i, column in enumerate(moves):
            drop_disc(board, column, HUMAN if i % 2 == 0 else COMPUTER)
        before = [row[:] for row in board]
        column, _, stats = choose_computer_move(board)
        check(column == target, label)
        check(board == before, "search restores board")
        check(1 <= stats.nodes <= 400, "three-ply node bound")

    # A full, balanced, gravity-valid draw with no four-in-a-row.
    draw = [[int(v) for v in row] for row in (
        "1122112", "2211221", "1122112", "2211221", "1122112", "2211221"
    )]
    check(sum(row.count(HUMAN) for row in draw) == 21, "balanced draw")
    check(not has_won(draw, HUMAN) and not has_won(draw, COMPUTER), "draw no winner")
    column, score, _ = choose_computer_move(draw)
    check(column is None and score == 0, "draw terminal score")

    def exhaustive(board, depth, maximizing):
        if has_won(board, COMPUTER):
            return WIN_SCORE + depth
        if has_won(board, HUMAN):
            return -WIN_SCORE - depth
        moves = legal_columns(board)
        if not moves:
            return 0
        if depth == 0:
            return evaluate(board)
        scores = []
        for c in moves:
            r = drop_disc(board, c, COMPUTER if maximizing else HUMAN)
            scores.append(exhaustive(board, depth - 1, not maximizing))
            board[r][c] = EMPTY
        return (max if maximizing else min)(scores)

    # Compare pruned search against an unpruned minimax reference.
    rng = Random(2025)
    positions = []
    for count in (1, 3, 5, 7, 9, 11, 13, 15):
        while True:
            board = new_board()
            for i in range(count):
                drop_disc(board, rng.choice(legal_columns(board)), HUMAN if i % 2 == 0 else COMPUTER)
                if has_won(board, HUMAN) or has_won(board, COMPUTER):
                    break
            else:
                positions.append(board)
                break
    for board in positions:
        before = [row[:] for row in board]
        column, score, _ = choose_computer_move(board)
        expected = exhaustive(board, SEARCH_DEPTH, True)
        check(score == expected and column in legal_columns(board), "pruned equals exhaustive")
        r = drop_disc(board, column, COMPUTER)
        check(exhaustive(board, SEARCH_DEPTH - 1, False) == expected, "chosen move optimal at depth 3")
        board[r][column] = EMPTY
        check(board == before, "reference and search restore board")

    # Exercise the actual game loop with an injected human input function.
    replies = iter(["4", "q"])
    transcript = StringIO()
    with redirect_stdout(transcript):
        result, board = play_game(lambda prompt: next(replies))
    check(result == "quit" and sum(v != EMPTY for row in board for v in row) == 2,
          "game alternates human and computer")
    check(transcript.getvalue().count("+---------------+") == 3, "board after each move")

    # A weak human strategy produces a complete game and a winner announcement.
    with redirect_stdout(StringIO()) as transcript:
        result, board = play_game(lambda prompt: "1")
    check(result == "computer" and has_won(board, COMPUTER), "complete game terminates on win")
    check("Computer wins!" in transcript.getvalue(), "winner announced")
    print(f"Passed {checks} checks, including 8 comparisons with exhaustive depth-3 minimax.")


run_checks()

Passed 52 checks, including 8 comparisons with exhaustive depth-3 minimax.


In [8]:
empty = new_board()
col, score, stats = choose_computer_move(empty)
print(f"Empty-board move: column {col + 1}")
print(f"Visited nodes: {stats.nodes} out of an unpruned maximum of 400")
print(f"Evaluated terminal or horizon leaves: {stats.leaves}")
print(f"Alpha-beta cutoffs: {stats.cutoffs}")

Empty-board move: column 4
Visited nodes: 76 out of an unpruned maximum of 400
Evaluated terminal or horizon leaves: 55
Alpha-beta cutoffs: 12


## Part (b): Complete sample game

This recorded example supplies column 1 as a deliberately weak human strategy.
It uses the same input function boundary and game loop as the interactive game.
The computer calculates every move with depth-3 Minimax. The interactive cell
below replaces this recorded input with the human's keyboard input.

In [9]:
sample_result, sample_final_board = play_game(input_fn=lambda prompt: "1")

Connect Four: you are X; the computer is O. You move first.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
+---------------+
You chose column 1.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| X . . . . . . |
+---------------+
Computer chose column 4 (depth 3, nodes 76, score 22).
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| X . . O . . . |
+---------------+
You chose column 1.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| X . . . . . . |
| X . . O . . . |
+---------------+
Computer chose column 5 (depth 3, nodes 167, score 95).
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| X . . . . . . |
| X . . O O . . |
+---------------+
You chose column 1.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . .

## Limits and conclusion

The implementation satisfies the required board representation, gravity,
dynamic human input, alternating turns, win and draw checks, board display and
fixed-depth Minimax. Alpha-beta pruning, precomputed windows and reversible
moves reduce avoidable work without changing the depth-3 result.

A depth of 3 does not guarantee optimal Connect Four play. Threats that resolve
beyond the horizon can be missed, and fixed weights only approximate position
quality. The simple text interface keeps the focus on the game logic and search.

## Part (b): Play against the computer

Run this final cell and respond to the input prompt. Enter q to end the game.

In [10]:
play_game()

Connect Four: you are X; the computer is O. You move first.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
+---------------+
You chose column 4.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . X . . . |
+---------------+
Computer chose column 4 (depth 3, nodes 79, score 4).
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . O . . . |
| . . . X . . . |
+---------------+
You chose column 5.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . O . . . |
| . . . X X . . |
+---------------+
Computer chose column 3 (depth 3, nodes 113, score 18).
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . O . . . |
| . . O X X . . |
+---------------+
You chose column 6.
  1 2 3 4 5 6 7
| . . . . . . . |
| . . . . . . . |
| . . . 

('computer',
 [[0, 0, 0, 1, 0, 2, 0],
  [2, 0, 1, 2, 2, 1, 0],
  [2, 0, 1, 2, 1, 2, 0],
  [1, 0, 2, 1, 1, 2, 1],
  [1, 0, 2, 2, 2, 1, 1],
  [2, 0, 2, 1, 1, 1, 2]])